In [17]:
# ============================================
# CELLULE 1 : IMPORTS
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [18]:
# ============================================
# CELLULE 2 : CHARGEMENT DES PRÉDICTIONS PROPHET
# ============================================

prophet_df = spark.read.format("delta").load(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/prophet_predictions/"
).toPandas()

prophet_df["ds"] = pd.to_datetime(prophet_df["ds"])

print(f"Prophet : {len(prophet_df)} lignes")
print(prophet_df.head())

In [19]:
# ============================================
# CELLULE 3 : CHARGEMENT DES PRÉDICTIONS LSTM
# ============================================

lstm_df = spark.read.format("delta").load(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/lstm_predictions/"
).toPandas()

lstm_df["ds"] = pd.to_datetime(lstm_df["ds"])

print(f"LSTM : {len(lstm_df)} lignes")
print(lstm_df.head())

In [21]:
# ============================================
# CELLULE 4 : TABLEAU COMPARATIF DES MÉTRIQUES
# ============================================

# Imports manquants
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Métriques Prophet 
prophet_mae = 0.021248517202228256
prophet_rmse = 0.02593587093822413

# Métriques LSTM 
lstm_mae = mean_absolute_error(lstm_df["y"], lstm_df["yhat"])
lstm_rmse = np.sqrt(mean_squared_error(lstm_df["y"], lstm_df["yhat"]))

comparison = pd.DataFrame({
    "Modèle": ["Prophet", "LSTM"],
    "MAE": [prophet_mae, lstm_mae],
    "RMSE": [prophet_rmse, lstm_rmse]
})

print("=" * 50)
print(" COMPARAISON PROPHET vs LSTM")
print("=" * 50)
print(comparison.to_string(index=False))
print("=" * 50)

# Calcul des différences
mae_diff = ((prophet_mae - lstm_mae) / prophet_mae) * 100
rmse_diff = ((prophet_rmse - lstm_rmse) / prophet_rmse) * 100

print(f"\n LSTM est meilleur de :")
print(f"   MAE  : -{mae_diff:.1f}%")
print(f"   RMSE : -{rmse_diff:.1f}%")

In [23]:
# ============================================
# CELLULE : ALIGNEMENT DES DONNÉES
# ============================================

# Fusionner sur la date pour avoir les deux prédictions côte à côte
merged = prophet_df[["ds", "y", "yhat"]].merge(
    lstm_df[["ds", "yhat"]],
    on="ds",
    suffixes=("_prophet", "_lstm")
)

# Vérifier l'alignement
print(f"Points alignés : {len(merged)}")
print(merged.head())

# Échantillon pour le graphique
sample = merged.iloc[::24]

In [24]:
# ============================================
# CELLULE 5 : GRAPHIQUE COMPARATIF 
# ============================================

plt.figure(figsize=(16, 8))

# Échantillon
sample = merged.iloc[::24]

# Réel (noir)
plt.plot(sample["ds"], sample["y"], 
         label="Réel", color="black", linewidth=2.5, zorder=3)

# Prophet (orange)
plt.plot(sample["ds"], sample["yhat_prophet"], 
         label="Prophet", color="orange", linewidth=2, alpha=0.8, zorder=2)

# LSTM (bleu)
plt.plot(sample["ds"], sample["yhat_lstm"], 
         label="LSTM", color="blue", linewidth=2, alpha=0.8, zorder=1)

plt.title("Comparaison Prophet vs LSTM\nPrédictions vs Valeurs Réelles", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Consommation (kWh)", fontsize=12)
plt.legend(fontsize=11, loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [25]:
# ============================================
# CELLULE 6 : GRAPHIQUE DES ERREURS
# ============================================

prophet_df["error"] = prophet_df["y"] - prophet_df["yhat"]
lstm_df["error"] = lstm_df["y"] - lstm_df["yhat"]

plt.figure(figsize=(14, 6))

plt.hist(prophet_df["error"], bins=50, alpha=0.5, label="Prophet", color="orange", density=True)
plt.hist(lstm_df["error"], bins=50, alpha=0.5, label="LSTM", color="blue", density=True)

plt.axvline(x=0, color="black", linestyle="--", linewidth=1)
plt.title("Distribution des Erreurs de Prédiction")
plt.xlabel("Erreur (Réel - Prédit)")
plt.ylabel("Densité")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [26]:
# ============================================
# CELLULE 7 : PRÉPARATION POWER BI
# ============================================

# Table des métriques
metrics_df = pd.DataFrame({
    "model": ["Prophet", "LSTM"],
    "mae": [0.021248517202228256, 0.013163],
    "rmse": [0.02593587093822413, 0.017093]
})

# Table des prédictions combinées
prophet_pred = prophet_df[["ds", "y", "yhat"]].copy()
prophet_pred["model"] = "Prophet"

lstm_pred = lstm_df[["ds", "y", "yhat"]].copy()
lstm_pred["model"] = "LSTM"

all_predictions = pd.concat([prophet_pred, lstm_pred], ignore_index=True)

# Sauvegarder
spark.createDataFrame(metrics_df).write.format("delta").mode("overwrite").save(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/powerbi_metrics/"
)

spark.createDataFrame(all_predictions).write.format("delta").mode("overwrite").save(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/powerbi_predictions_finale/"
)

print(" Données Power BI prêtes")

In [27]:
# ============================================
# CELLULE 8 : SAUVEGARDE RÉSULTATS COMPARATIFS
# ============================================

# Sauvegarder le tableau comparatif
spark.createDataFrame(comparison).write.format("delta").mode("overwrite").save(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/comparison_results/"
)

print(" Tableau comparatif sauvegardé")